In [ ]:
from pathlib import Path
import os, json, pandas as pd
from docx import Document

print("PLAN D’ANALYSE")
print("1. Vérifier le bassin Vatencul existant, sa surface, sa géométrie et son exutoire.")
print("2. Comme le seuil demandé est 70 ha, conserver la délimitation aval si la surface contrôlée est >= 70 ha; sinon prolonger.")
print("3. Découper et analyser les données disponibles: MNT/MNS/MNH, BD TOPO, MOS, PLU, PPRI et imperméabilisation.")
print("4. Produire les diagnostics topographiques et objets préparatoires réalisables sans modèle hydraulique dynamique.")
print("5. Construire un paquet de données documenté pour HEC-RAS/GeoServer/MapStore et une version révisée du mémoire.")
print("6. Contrôler les hypothèses, la qualité, les unités, les référentiels et expliciter les limites.")

root=Path('/workspace')
out=root/'vatencul_lidar_websig_outputs'
out.mkdir(exist_ok=True)
print(f"\nDossier de sortie: {out}")
print("Mémoire présent:", (root/'Memoire_NAOMIS_Final_V3_revise.docx').exists())
print("Bassin présent:", (root/'BASSIN_VATENCUL_FINAL_BASE.gpkg').exists())
print("Nombre de MNT 0,5 m:", len(list(root.glob('LHD_*_MNT_O_0M50_*.tif'))))
print("Nombre de MNS 0,5 m:", len(list(root.glob('LHD_*_MNS_O_0M50_*.tif'))))
print("Nombre de MNH 0,5 m:", len(list(root.glob('LHD_*_MNH_O_0M50_*.tif'))))

In [ ]:
import sys, subprocess, importlib.util
needed=['geopandas','rasterio','fiona','shapely']
missing=[p for p in needed if importlib.util.find_spec(p) is None]
if missing:
    subprocess.check_call([sys.executable,'-m','pip','install','-q',*missing])

from pathlib import Path
import geopandas as gpd, pandas as pd, numpy as np, rasterio
import fiona
from shapely.geometry import Point
root=Path('/workspace')
basin_path=root/'BASSIN_VATENCUL_FINAL_BASE.gpkg'
print('Couches bassin:', fiona.listlayers(basin_path))
for lyr in fiona.listlayers(basin_path):
    g=gpd.read_file(basin_path, layer=lyr)
    print('\n',lyr, 'shape=',g.shape,'CRS=',g.crs,'types=',g.geom_type.value_counts().to_dict())
    if g.crs and g.crs.is_projected:
        print('surface totale ha=',g.geometry.area.sum()/10000,'longueur totale km=',g.geometry.length.sum()/1000)
    print(g.drop(columns='geometry').head().to_string())

b=gpd.read_file(root/'BASSIN_VATENCUL_BASE.shp')
print('\nSHP CRS',b.crs,'entités',len(b),'valides',b.geometry.is_valid.sum(),'surface_ha',b.area.sum()/1e4)
print('bounds',b.total_bounds)
outlet=Point(643225.25,6844872.75)
union=b.geometry.union_all()
print('exutoire couvert=',union.covers(outlet),'distance à limite m=',union.boundary.distance(outlet))

rows=[]
for p in sorted(root.glob('LHD_*_MNT_O_0M50_*.tif')):
    with rasterio.open(p) as ds:
        rows.append([p.name,ds.crs.to_string(),ds.res,ds.bounds.left,ds.bounds.bottom,ds.bounds.right,ds.bounds.top,ds.nodata])
rd=pd.DataFrame(rows,columns=['fichier','crs','resolution','xmin','ymin','xmax','ymax','nodata'])
print('\nMNT:')
print(rd.to_string(index=False))

In [ ]:
from pathlib import Path
import fiona, geopandas as gpd, pandas as pd, json
root=Path('/workspace')
# Inventaire ciblé des couches disponibles et de leur schéma
for path in [root/'BDT_3-5_GPKG_LAMB93_D091-ED2026-06-15.gpkg', root/'BASSIN_VATENCUL_FINAL_BASE.gpkg']:
    print('\n',path.name)
    ls=fiona.listlayers(path)
    print('nombre de couches:',len(ls))
    print(ls)

mos=gpd.read_file(root/'mos_villebon.geojson')
print('\nMOS shape/crs:',mos.shape,mos.crs)
print('colonnes:',mos.columns.tolist())
print(mos.drop(columns='geometry').head().to_string())

# Repère les couches BD TOPO pertinentes par nom
layers=fiona.listlayers(root/'BDT_3-5_GPKG_LAMB93_D091-ED2026-06-15.gpkg')
keywords=['BATIMENT','ROUTE','TRONCON','HYDRO','COURS','HAIE','TALUS','VEGETATION','SURFACE']
sel=[x for x in layers if any(k in x.upper() for k in keywords)]
print('\nCouches candidates:',sel)

In [ ]:
import sys, subprocess, importlib.util
pkgs=['pyflwdir','scipy','matplotlib','pyogrio']
missing=[p for p in pkgs if importlib.util.find_spec(p) is None]
if missing:
    subprocess.check_call([sys.executable,'-m','pip','install','-q',*missing])
import pyflwdir, inspect
print('pyflwdir',pyflwdir.__version__)
print('from_dem:',inspect.signature(pyflwdir.from_dem))
print(pyflwdir.from_dem.__doc__[:1200])

In [ ]:
from docx import Document
from pathlib import Path
import re
p=Path('/workspace/Memoire_NAOMIS_Final_V3_revise.docx')
doc=Document(p)
paras=[x.text.strip() for x in doc.paragraphs]
# Locate all paragraphs mentioning key parameters
terms=['hydrocondition','burn','breach','D8','seuil d’accumulation','seuil d\'accumulation','MapStore','Vatencul']
for i,t in enumerate(paras):
    low=t.lower()
    if any(k.lower() in low for k in terms) and t:
        print(f'{i}: {t[:500]}')

In [ ]:
from docx import Document
from pathlib import Path
p=Path('/workspace/Memoire_NAOMIS_Final_V3_revise.docx')
doc=Document(p)
paras=[x.text.strip() for x in doc.paragraphs]
for a,b in [(360,470),(545,645),(668,690)]:
    print(f'\n--- PARAGRAPHES {a}-{b} ---')
    for i in range(a,b):
        if paras[i]: print(f'{i}: {paras[i]}')

In [ ]:
from pathlib import Path
import numpy as np, pandas as pd, geopandas as gpd, rasterio, fiona, json, os
from rasterio.merge import merge
from rasterio.mask import mask
from rasterio.features import geometry_mask
from shapely.geometry import mapping
from scipy import ndimage

root=Path('/workspace'); out=root/'vatencul_lidar_websig_outputs'
rasdir=out/'rasters'; vecdir=out/'vectors'; tabdir=out/'tables'; docdir=out/'documentation'; webdir=out/'mapstore_geoserver'
for d in [rasdir,vecdir,tabdir,docdir,webdir]: d.mkdir(parents=True,exist_ok=True)

basin=gpd.read_file(root/'BASSIN_VATENCUL_FINAL_BASE.gpkg',layer='bassin').to_crs(2154)
geom=basin.geometry.union_all()
clipgeom=geom.buffer(100)

# mosaïque et découpe MNT/MNS/MNH sur bassin + 100 m, en conservant la grille IGN
outputs={}
for prod in ['MNT','MNS','MNH']:
    files=sorted(root.glob(f'LHD_*_{prod}_O_0M50_*.tif'))
    srcs=[rasterio.open(p) for p in files]
    mosaic, trans=merge(srcs,bounds=clipgeom.bounds,nodata=-9999.0)
    profile=srcs[0].profile.copy()
    profile.update(height=mosaic.shape[1],width=mosaic.shape[2],transform=trans,count=1,
                   compress='DEFLATE',tiled=True,blockxsize=512,blockysize=512,predictor=3,
                   BIGTIFF='IF_SAFER',nodata=-9999.0)
    arr=mosaic[0].astype('float32')
    inside=geometry_mask([mapping(clipgeom)],out_shape=arr.shape,transform=trans,invert=True)
    arr[~inside]=-9999.0
    dst=rasdir/f'{prod.lower()}_lidar_0p50m_bassin_plus100m.tif'
    with rasterio.open(dst,'w',**profile) as ds:
        ds.write(arr,1)
        ds.update_tags(source='IGN LiDAR HD',vertical_datum='IGN69',processing='mosaic and polygon clip; no elevation alteration')
    for s in srcs:s.close()
    outputs[prod]=(dst,arr,profile)
    valid=arr!=-9999
    print(prod,dst.name,arr.shape,'valid cells',int(valid.sum()),'range',float(arr[valid].min()),float(arr[valid].max()))

# Dérivés strictement topographiques sur MNT brut
mnt=outputs['MNT'][1]; profile=outputs['MNT'][2]; valid=mnt!=-9999
z=np.where(valid,mnt,np.nan)
# remplissage local uniquement pour calcul gradients aux bordures NoData
filled=np.where(valid,mnt,ndimage.generic_filter(np.where(valid,mnt,np.nan),np.nanmean,size=3,mode='nearest'))
dzdy,dzdx=np.gradient(filled,0.5,0.5)
slope=np.degrees(np.arctan(np.hypot(dzdx,dzdy))).astype('float32'); slope[~valid]=-9999
aspect=(np.degrees(np.arctan2(-dzdx,dzdy))+360)%360; aspect=aspect.astype('float32'); aspect[~valid]=-9999
az=np.deg2rad(315); alt=np.deg2rad(45); sl=np.arctan(np.hypot(dzdx,dzdy)); asp=np.deg2rad(aspect)
hill=(255*(np.sin(alt)*np.cos(sl)+np.cos(alt)*np.sin(sl)*np.cos(az-asp))).clip(0,255).astype('float32'); hill[~valid]=-9999
# courbure laplacienne indicative (m-1 approx.)
curv=ndimage.laplace(filled,mode='nearest')/(0.5**2); curv=curv.astype('float32'); curv[~valid]=-9999
for name,arr,unit in [('pente_degres',slope,'degree'),('aspect_degres',aspect,'degree'),('ombrage_315_45',hill,'0-255'),('courbure_laplacienne',curv,'m-1')]:
    prof=profile.copy(); prof.update(dtype='float32',nodata=-9999.0,compress='DEFLATE',predictor=3)
    p=rasdir/f'{name}.tif'
    with rasterio.open(p,'w',**prof) as ds:
        ds.write(arr,1); ds.update_tags(source='MNT LiDAR HD brut',unit=unit)
    print('écrit',p.name)

# Contrôles descriptifs strictement dans le bassin (pas buffer)
inside_basin=geometry_mask([mapping(geom)],out_shape=mnt.shape,transform=profile['transform'],invert=True)&valid
vals=mnt[inside_basin]; svals=slope[inside_basin]; mnh=outputs['MNH'][1][inside_basin]
summary=pd.DataFrame({
 'variable':['altitude_m','pente_deg','mnh_m'],
 'n':[vals.size,svals.size,mnh.size],
 'min':[np.nanmin(vals),np.nanmin(svals),np.nanmin(mnh)],
 'q25':[np.nanpercentile(vals,25),np.nanpercentile(svals,25),np.nanpercentile(mnh,25)],
 'mediane':[np.nanmedian(vals),np.nanmedian(svals),np.nanmedian(mnh)],
 'moyenne':[np.nanmean(vals),np.nanmean(svals),np.nanmean(mnh)],
 'q75':[np.nanpercentile(vals,75),np.nanpercentile(svals,75),np.nanpercentile(mnh,75)],
 'max':[np.nanmax(vals),np.nanmax(svals),np.nanmax(mnh)]})
summary.to_csv(tabdir/'statistiques_relief_bassin.csv',index=False)
print(summary.to_string(index=False))

In [ ]:
import pyflwdir, inspect
# Méthodes nécessaires, vérifiées avant calcul lourd
from pyflwdir import dem
print('fill_depressions',inspect.signature(dem.fill_depressions))
# petit objet factice pour signatures
x=np.array([[3,2,1],[4,3,2],[5,4,3]],dtype=np.float32)
f=pyflwdir.from_dem(x,nodata=-9999,transform=rasterio.Affine(1,0,0,0,-1,3))
for m in ['accuflux','upstream_area','basins','streams','snap']:
    fn=getattr(f,m,None)
    print(m, inspect.signature(fn) if fn else None)

In [ ]:
import time, numpy as np, rasterio, pyflwdir
from pyflwdir import dem
from rasterio.transform import rowcol

mnt=outputs['MNT'][1].astype('float32'); profile=outputs['MNT'][2]; transform=profile['transform']
t0=time.time()
# Diagnostic indépendant sur MNT brut: remplissage généralisé, donc distinct de la délimitation officielle hydroconditionnée
filled_dem, d8 = dem.fill_depressions(mnt, outlets='edge', nodata=-9999.0, max_depth=-1.0, connectivity=8)
print('fill seconds',round(time.time()-t0,1))
flw=pyflwdir.from_array(d8, ftype='d8', transform=transform, latlon=False, check_ftype=False)
upa=flw.upstream_area(unit='m2').astype('float32')
valid=mnt!=-9999.0
filldepth=np.where(valid,filled_dem-mnt,-9999.0).astype('float32')
upa[~valid]=-9999.0
# Raster D8 ESRI coding, aire contributive et profondeur de remplissage
for name,arr,dtype,nodata,unit in [
 ('mnt_rempli_diagnostic',filled_dem.astype('float32'),'float32',-9999.0,'m'),
 ('direction_d8_diagnostic',d8.astype('uint8'),'uint8',247,'ESRI_D8'),
 ('aire_contributive_d8_m2',upa,'float32',-9999.0,'m2'),
 ('profondeur_remplissage_depressions_m',filldepth,'float32',-9999.0,'m')]:
    prof=profile.copy(); prof.update(dtype=dtype,nodata=nodata,compress='DEFLATE',predictor=(3 if dtype=='float32' else 2))
    p=rasdir/f'{name}.tif'
    with rasterio.open(p,'w',**prof) as ds:
        ds.write(arr,1)
        ds.update_tags(method='D8 on raw IGN LiDAR DEM after generalized depression filling; screening only',unit=unit,
                       limitation='Not culvert-conditioned; not a hydraulic simulation')
    print('écrit',p.name)

# Contrôle au point exutoire documenté
rr,cc=rowcol(transform,643225.25,6844872.75)
print('outlet rc',rr,cc,'raw uparea ha',upa[rr,cc]/1e4,'elev',mnt[rr,cc])
# volume de remplissage dans bassin, présenté comme plafond topographique non stockage réel
inside=inside_basin
pos=(filldepth>0)&inside
stats={'cells_basin':int(inside.sum()),'cells_raised':int(pos.sum()),'fraction_raised_pct':float(100*pos.sum()/inside.sum()),
       'fill_depth_median_positive_m':float(np.median(filldepth[pos])) if pos.any() else 0,
       'fill_depth_p95_positive_m':float(np.percentile(filldepth[pos],95)) if pos.any() else 0,
       'fill_depth_max_m':float(filldepth[pos].max()) if pos.any() else 0,
       'fill_volume_upper_bound_m3':float(filldepth[pos].sum()*0.25)}
print(json.dumps(stats,indent=2))
pd.DataFrame([stats]).to_csv(tabdir/'diagnostic_depressions_remplissage.csv',index=False)
print('total seconds',round(time.time()-t0,1))

In [ ]:
import geopandas as gpd, fiona, pandas as pd
bd=root/'BDT_3-5_GPKG_LAMB93_D091-ED2026-06-15.gpkg'
for lyr in ['batiment','troncon_de_route','cours_d_eau','troncon_hydrographique','haie','zone_de_vegetation','construction_lineaire','detail_hydrographique']:
    g=gpd.read_file(bd,layer=lyr,bbox=geom.buffer(100).bounds,engine='pyogrio')
    print('\n',lyr,'n',len(g),'crs',g.crs,'cols',g.columns.tolist())
    print(g.drop(columns='geometry').head(2).to_string(index=False))

In [ ]:
import geopandas as gpd, pandas as pd, numpy as np, rasterio, json, warnings
from shapely.geometry import shape, mapping
from rasterio.features import shapes
warnings.filterwarnings('ignore', category=UserWarning)

bd=root/'BDT_3-5_GPKG_LAMB93_D091-ED2026-06-15.gpkg'
# Copie des couches de référence du bassin dans un GeoPackage de livraison
pkg=vecdir/'vatencul_objets_websig.gpkg'
if pkg.exists(): pkg.unlink()
for lyr in fiona.listlayers(root/'BASSIN_VATENCUL_FINAL_BASE.gpkg'):
    g=gpd.read_file(root/'BASSIN_VATENCUL_FINAL_BASE.gpkg',layer=lyr)
    g.to_file(pkg,layer=lyr,driver='GPKG')

# Axes de concentration topographique sur MNT brut/rempli, seuil repris du mémoire: 5000 m²
stream_mask=(upa>=5000)&inside_basin
# réseau pyflwdir, puis découpe bassin
features=flw.streams(mask=stream_mask)
axes=gpd.GeoDataFrame.from_features(features,crs='EPSG:2154')
if len(axes):
    axes=axes[axes.geometry.notna()].copy()
    axes['geometry']=axes.geometry.intersection(geom)
    axes=axes[~axes.geometry.is_empty].explode(index_parts=False).reset_index(drop=True)
    axes['id_axe']=np.arange(1,len(axes)+1)
    axes['longueur_m']=axes.length
    axes['seuil_m2']=5000
    axes['statut']='diagnostic_MNT_brut_rempli'
    axes['limite']='Sans hydroconditionnement des buses; axe topographique, non débit'
    axes.to_file(pkg,layer='axes_concentration_brut_5000m2',driver='GPKG')
print('axes:',len(axes),'longueur km',axes.length.sum()/1000 if len(axes) else 0)

# Dépressions significatives pour inspection: profondeur >=0,10 m et aire connectée >=10 m²
maskdep=(filldepth>=0.10)&inside_basin
lab,nlab=ndimage.label(maskdep,structure=np.ones((3,3),dtype=int))
counts=np.bincount(lab.ravel())
keep_ids=np.where(counts>=40)[0]  # 40 cellules * 0,25 = 10 m²
keep=np.isin(lab,keep_ids[keep_ids!=0])
recs=[]
for geomj,val in shapes(lab.astype('int32'),mask=keep,transform=transform,connectivity=8):
    lid=int(val)
    if lid==0 or counts[lid]<40: continue
    pix=(lab==lid)
    depths=filldepth[pix]
    recs.append({'id_dep':lid,'area_m2':counts[lid]*0.25,'volume_fill_m3':float(depths.sum()*0.25),
                 'depth_max_m':float(depths.max()),'depth_med_m':float(np.median(depths)),
                 'statut':'candidat_topographique','confiance':'faible_sans_controle_terrain',
                 'geometry':shape(geomj)})
deps=gpd.GeoDataFrame(recs,crs=2154)
if len(deps):
    deps=deps.sort_values('volume_fill_m3',ascending=False).reset_index(drop=True)
    deps['rang_volume']=np.arange(1,len(deps)+1)
    deps.to_file(pkg,layer='depressions_candidates',driver='GPKG')
print('dépressions candidates:',len(deps),'aire ha',deps.area.sum()/1e4 if len(deps) else 0,'volume fill m3',deps.volume_fill_m3.sum() if len(deps) else 0)

# BD TOPO découpée bassin (+ seulement objets réellement dans le bassin)
def read_clip(layer, buffer=0):
    zone=geom.buffer(buffer)
    g=gpd.read_file(bd,layer=layer,bbox=zone.bounds,engine='pyogrio')
    g=g[g.intersects(zone)].copy()
    g['geometry']=g.geometry.intersection(zone)
    return g[~g.geometry.is_empty]

bati=read_clip('batiment')
routes=read_clip('troncon_de_route')
hydro=read_clip('troncon_hydrographique')
vege=read_clip('zone_de_vegetation')
haies=read_clip('haie')
ponts=read_clip('construction_lineaire')
if 'nature' in ponts: ponts=ponts[ponts['nature'].astype(str).str.contains('Pont',case=False,na=False)].copy()

# Attributs d'inspection sans conclure à l'inondation
if len(axes):
    bati['dist_axe_brut_m']=bati.geometry.distance(axes.geometry.union_all())
    bati['proche_axe_10m']=bati.dist_axe_brut_m<=10
    routes['croise_axe_brut']=routes.intersects(axes.geometry.union_all())
else:
    bati['dist_axe_brut_m']=np.nan; bati['proche_axe_10m']=False; routes['croise_axe_brut']=False
bati['interpretation']='proximité topographique; ne prouve pas une inondation du bâtiment'
routes['interpretation']='croisement topographique à inspecter; ne prouve pas une buse'
for layer,g in [('batiments',bati),('routes',routes),('hydrographie_bdtopo',hydro),('vegetation_bdtopo',vege),('haies_bdtopo',haies),('ponts_bdtopo',ponts)]:
    if len(g): g.to_file(pkg,layer=layer,driver='GPKG')

metrics={'surface_bassin_ha':geom.area/1e4,'axes_brut_longueur_km':axes.length.sum()/1000 if len(axes) else 0,
         'batiments_bassin_n':len(bati),'batiments_proches_axes_brut_10m_n':int(bati.proche_axe_10m.sum()),
         'routes_bassin_n':len(routes),'troncons_routes_croisant_axes_brut_n':int(routes.croise_axe_brut.sum()),
         'depressions_candidates_n':len(deps),'ponts_bdtopo_n':len(ponts),'troncons_hydro_bdtopo_n':len(hydro)}
pd.DataFrame([metrics]).to_csv(tabdir/'indicateurs_territoriaux.csv',index=False)
print(json.dumps(metrics,indent=2,ensure_ascii=False))

In [ ]:
import pandas as pd, numpy as np, geopandas as gpd
from math import radians
files=sorted(root.glob('MN_91_*.csv.gz'))
station_parts=[]
for p in files:
    for ch in pd.read_csv(p,sep=';',compression='gzip',usecols=['NUM_POSTE','NOM_USUEL','LAT','LON','ALTI','AAAAMMJJHHMN','RR','QRR'],chunksize=500000,low_memory=False):
        ch['date']=pd.to_datetime(ch['AAAAMMJJHHMN'].astype(str),format='%Y%m%d%H%M',errors='coerce')
        gr=ch.groupby(['NUM_POSTE','NOM_USUEL','LAT','LON','ALTI'],dropna=False).agg(debut=('date','min'),fin=('date','max'),n=('date','size'),n_rr=('RR','count')).reset_index()
        station_parts.append(gr)
st=pd.concat(station_parts).groupby(['NUM_POSTE','NOM_USUEL','LAT','LON','ALTI'],dropna=False).agg(debut=('debut','min'),fin=('fin','max'),n=('n','sum'),n_rr=('n_rr','sum')).reset_index()
cent=gpd.GeoSeries([geom.centroid],crs=2154).to_crs(4326).iloc[0]
def hav(lat,lon):
    R=6371.; p1=np.radians(cent.y);p2=np.radians(lat.astype(float));dp=p2-p1;dl=np.radians(lon.astype(float)-cent.x)
    return 2*R*np.arcsin(np.sqrt(np.sin(dp/2)**2+np.cos(p1)*np.cos(p2)*np.sin(dl/2)**2))
st['distance_bassin_km']=hav(st.LAT,st.LON)
st['completude_rr_pct']=100*st.n_rr/st.n
st=st.sort_values('distance_bassin_km')
print(st.to_string(index=False))
st.to_csv(tabdir/'stations_meteo_6min_essonne.csv',index=False)

In [ ]:
import pandas as pd, numpy as np, json
station=91275001
parts=[]
for p in sorted(root.glob('MN_91_*.csv.gz')):
    ch=pd.read_csv(p,sep=';',compression='gzip',usecols=['NUM_POSTE','NOM_USUEL','AAAAMMJJHHMN','RR','QRR'],low_memory=False)
    ch=ch[ch.NUM_POSTE==station].copy()
    if len(ch): parts.append(ch)
rain=pd.concat(parts,ignore_index=True).drop_duplicates(['AAAAMMJJHHMN']).sort_values('AAAAMMJJHHMN')
rain['datetime_utc']=pd.to_datetime(rain.AAAAMMJJHHMN.astype(str),format='%Y%m%d%H%M',errors='coerce')
rain=rain.set_index('datetime_utc').sort_index()
print('Gometz lignes',len(rain),'du',rain.index.min(),'au',rain.index.max())
print('QRR:',rain.QRR.value_counts(dropna=False).sort_index().to_dict(),'RR manquants',rain.RR.isna().sum())
# grille complète 6 min; trous explicités, pas imputés
series=rain.RR.reindex(pd.date_range(rain.index.min(),rain.index.max(),freq='6min',tz=None))
# cumuls roulants uniquement si fenêtre complète
rows=[]
for label,n in [('6min',1),('1h',10),('3h',30),('6h',60),('24h',240)]:
    roll=series.rolling(n,min_periods=n).sum()
    t=roll.idxmax(); rows.append({'duree':label,'pas_6min':n,'cumul_max_mm':float(roll.max()),'fin_utc':t,'debut_utc':t-pd.Timedelta(minutes=6*(n-1))})
extremes=pd.DataFrame(rows)
print(extremes.to_string(index=False))
extremes.to_csv(tabdir/'maxima_pluie_observee_gometz.csv',index=False)
# événement de démonstration: fenêtre de 24h maximisant le cumul, aucune interprétation de période de retour
end=pd.Timestamp(extremes.loc[extremes.duree=='24h','fin_utc'].iloc[0]); start=end-pd.Timedelta(hours=23,minutes=54)
event=rain.loc[start:end,['NUM_POSTE','NOM_USUEL','RR','QRR']].reset_index()
event['minutes_depuis_debut']=(event.datetime_utc-event.datetime_utc.min()).dt.total_seconds()/60
event['cumul_mm']=event.RR.fillna(0).cumsum() if event.RR.notna().all() else np.nan
event.to_csv(tabdir/'hyetogramme_historique_gometz_max24h.csv',index=False)
print('événement',start,end,'n',len(event),'cumul',event.RR.sum(),'manquants',event.RR.isna().sum())

In [ ]:
import geopandas as gpd, pandas as pd, numpy as np, rasterio, fiona, json, warnings
from rasterio.mask import mask as rio_mask
from rasterio.warp import reproject, Resampling
from shapely.geometry import mapping
warnings.filterwarnings('ignore')

# MOS 2025 découpé au bassin et synthèse surfacique
mos=gpd.read_file(root/'mos_villebon.geojson').to_crs(2154)
mos=mos[mos.intersects(geom)].copy(); mos['geometry']=mos.geometry.intersection(geom); mos=mos[~mos.geometry.is_empty]
mos['surface_ha']=mos.area/1e4
mos['part_bassin_pct']=100*mos.surface_ha/(geom.area/1e4)
mos.to_file(pkg,layer='mos_2025',driver='GPKG')
mos_summary=mos.groupby(['mos2025','2025poste'],dropna=False).surface_ha.sum().reset_index().sort_values('surface_ha',ascending=False)
mos_summary['part_bassin_pct']=100*mos_summary.surface_ha/(geom.area/1e4)
mos_summary.to_csv(tabdir/'occupation_sol_mos2025.csv',index=False)
print('MOS top:'); print(mos_summary.head(15).to_string(index=False))

# PLU et PPRI réellement intersectés
plu_dir=root/'91661_PLU_20250410'/'Donnees_geographiques'
plu_candidates=list(root.glob('91661_*_20250410.shp'))+list(plu_dir.glob('91661_*_20250410.shp'))
seen=set()
for p in plu_candidates:
    if p.stem in seen: continue
    seen.add(p.stem)
    g=gpd.read_file(p).to_crs(2154)
    g=g[g.intersects(geom)].copy()
    if len(g):
        g['geometry']=g.geometry.intersection(geom); g=g[~g.geometry.is_empty]
        lname='plu_'+p.stem.replace('91661_','').replace('_20250410','').lower()
        g.to_file(pkg,layer=lname[:62],driver='GPKG')
        print(lname,len(g))
ppri=gpd.read_file(root/'N_ZONE_REG_PPRN_20110001_S_091.shp').to_crs(2154)
ppri=ppri[ppri.intersects(geom)].copy()
if len(ppri):
    ppri['geometry']=ppri.geometry.intersection(geom); ppri=ppri[~ppri.geometry.is_empty]
    ppri['surface_ha']=ppri.area/1e4
    ppri.to_file(pkg,layer='ppri_yvette',driver='GPKG')
print('PPRI entités',len(ppri),'surface ha',ppri.area.sum()/1e4 if len(ppri) else 0)
if len(ppri): print(ppri[['CODEZONE','NOM','surface_ha']].to_string(index=False))

# Imperméabilisation Copernicus 2024: moyenne zonale, sans substitution au MOS
imd_path=root/'CLMS_NVLCC_IMD_S2024_R10m_E37N28_03035_V01_R01.tif'
with rasterio.open(imd_path) as src:
    # reprojette le masque bassin vers CRS raster, puis clip
    bg=basin.to_crs(src.crs)
    a,tr=rio_mask(src,[mapping(bg.geometry.union_all())],crop=True,filled=False)
    vals=a[0].compressed()
    print('IMD dtype/range/nodata',src.dtypes,src.nodata,(vals.min(),vals.max()),'n',len(vals))
    imd_stats={'n_pixels_valides':len(vals),'moyenne_pct':float(vals.mean()),'mediane_pct':float(np.median(vals)),
               'q25_pct':float(np.percentile(vals,25)),'q75_pct':float(np.percentile(vals,75)),
               'part_pixels_ge_80pct':float(100*np.mean(vals>=80))}
pd.DataFrame([imd_stats]).to_csv(tabdir/'impermeabilisation_copernicus_2024.csv',index=False)
print(json.dumps(imd_stats,indent=2))

In [ ]:
import matplotlib.pyplot as plt, matplotlib.colors as colors
from matplotlib.patches import Patch
import numpy as np, pandas as pd, geopandas as gpd
figdir=out/'figures'; figdir.mkdir(exist_ok=True)
plt.rcParams.update({'font.size':9,'axes.titlesize':12,'axes.labelsize':9})

# Carte synthétique terrain/axes/objets
fig,ax=plt.subplots(figsize=(9,9))
extent=[profile['transform'].c, profile['transform'].c+profile['transform'].a*mnt.shape[1],
        profile['transform'].f+profile['transform'].e*mnt.shape[0], profile['transform'].f]
hs=np.ma.masked_equal(hill,-9999)
ax.imshow(hs,cmap='gray',extent=extent,origin='upper',vmin=30,vmax=230)
basin.boundary.plot(ax=ax,color='#111111',linewidth=1.4,label='Bassin 73,06 ha')
if len(axes): axes.plot(ax=ax,color='#1676d2',linewidth=1.1,label='Axes D8, seuil 5 000 m²')
if len(deps): deps.head(30).boundary.plot(ax=ax,color='#8e44ad',linewidth=.7,label='30 dépressions au plus fort volume de remplissage')
gpd.read_file(root/'BASSIN_VATENCUL_FINAL_BASE.gpkg',layer='troncons_visibles').plot(ax=ax,color='#00bcd4',linewidth=2.2,label='Vatencul visible')
gpd.read_file(root/'BASSIN_VATENCUL_FINAL_BASE.gpkg',layer='raccords_souterrains').plot(ax=ax,color='#ff9800',linewidth=2,linestyle='--',label='Raccords fonctionnels documentés')
gpd.read_file(root/'BASSIN_VATENCUL_FINAL_BASE.gpkg',layer='exutoire').plot(ax=ax,color='red',markersize=35,label='Exutoire')
ax.set_xlim(geom.bounds[0]-30,geom.bounds[2]+30);ax.set_ylim(geom.bounds[1]-30,geom.bounds[3]+30)
ax.set_aspect('equal');ax.set_xlabel('Lambert-93 Est (m)');ax.set_ylabel('Lambert-93 Nord (m)')
ax.set_title('Bassin du Vatencul : relief et objets préparés pour le diagnostic')
ax.legend(loc='lower left',fontsize=7,framealpha=.9)
ax.text(.99,.01,'Fond : ombrage du MNT LiDAR HD IGN 0,50 m\nAxes sur MNT brut rempli : diagnostic, pas une simulation',transform=ax.transAxes,ha='right',va='bottom',fontsize=7,bbox=dict(facecolor='white',alpha=.8,edgecolor='none'))
fig.tight_layout(); fig.savefig(figdir/'carte_diagnostic_vatencul.png',dpi=220);plt.close(fig)

# Relief / hypsométrie / pente
fig,axs=plt.subplots(1,2,figsize=(11,4.2))
axs[0].hist(vals,bins=50,color='#4c78a8',edgecolor='none');axs[0].axvline(np.median(vals),color='black',ls='--',label=f'Médiane {np.median(vals):.1f} m');axs[0].set(xlabel='Altitude IGN69 (m)',ylabel='Nombre de cellules',title='Distribution altimétrique');axs[0].legend()
axs[1].hist(np.clip(svals,0,45),bins=np.arange(0,46,1),color='#f58518',edgecolor='none');axs[1].axvline(np.median(svals),color='black',ls='--',label=f'Médiane {np.median(svals):.1f}°');axs[1].set(xlabel='Pente (degrés, valeurs >45° regroupées)',ylabel='Nombre de cellules',title='Distribution des pentes');axs[1].legend()
fig.suptitle('Relief du bassin du Vatencul, MNT LiDAR HD 0,50 m');fig.tight_layout();fig.savefig(figdir/'relief_hypsometrie_pentes.png',dpi=220);plt.close(fig)

# MOS
plotmos=mos_summary.head(10).sort_values('surface_ha')
fig,ax=plt.subplots(figsize=(9,5));ax.barh(plotmos['2025poste'],plotmos.surface_ha,color='#54a24b');ax.set(xlabel='Surface dans le bassin (ha)',title='Dix principales occupations du sol, MOS 2025');
for y,v in enumerate(plotmos.surface_ha):ax.text(v+.12,y,f'{v:.1f} ha',va='center',fontsize=8)
fig.tight_layout();fig.savefig(figdir/'occupation_sol_mos2025.png',dpi=220,bbox_inches='tight');plt.close(fig)

# pluie
fig,ax=plt.subplots(figsize=(10,4));ax.bar(event.datetime_utc,event.RR,width=0.0035,color='#2c7fb8');ax.set(ylabel='Pluie par 6 min (mm)',xlabel='Date et heure UTC',title='Hyétogramme observé à Gometz-le-Châtel : maximum glissant 24 h de la série disponible');ax.text(.01,.95,f'Cumul = {event.RR.sum():.1f} mm en 24 h\nAucune lacune dans la fenêtre\nDistance au bassin : 6,58 km',transform=ax.transAxes,va='top',bbox=dict(facecolor='white',alpha=.85,edgecolor='#cccccc'))
fig.autofmt_xdate();fig.tight_layout();fig.savefig(figdir/'hyetogramme_gometz_2024_10_09.png',dpi=220);plt.close(fig)
print('Figures:',[p.name for p in figdir.glob('*.png')])

In [ ]:
from pathlib import Path
import json, shutil, hashlib, os, pandas as pd, geopandas as gpd, rasterio
from rasterio.shutil import copy as rio_copy

# Paquet HEC-RAS préparatoire, sans paramètres hydrauliques inventés
hec=out/'hec_ras_preparation'; hec.mkdir(exist_ok=True)
# Terrain brut, limite, axes connus et occupation du sol
for src,dst in [
 (rasdir/'mnt_lidar_0p50m_bassin_plus100m.tif',hec/'terrain_mnt_lidar_0p50m.tif'),
 (tabdir/'hyetogramme_historique_gometz_max24h.csv',hec/'pluie_observee_gometz_max24h.csv')]:
    shutil.copy2(src,dst)
for lyr in ['bassin','exutoire','troncons_visibles','raccords_souterrains','mos_2025','batiments','routes','ponts_bdtopo']:
    try:
        g=gpd.read_file(pkg,layer=lyr)
        g.to_file(hec/'geometries_preparatoires.gpkg',layer=lyr,driver='GPKG')
    except Exception as e: print('skip',lyr,e)

hec_manifest=pd.DataFrame([
 ['Terrain','terrain_mnt_lidar_0p50m.tif','prêt','MNT brut IGN69; hydroconditionnement HEC-RAS non appliqué'],
 ['Domaine 2D','geometries_preparatoires.gpkg:bassin','prêt à adapter','La limite hydrologique n’est pas automatiquement la limite de calcul hydraulique'],
 ['Exutoire','geometries_preparatoires.gpkg:exutoire','repère','Condition aval non définie'],
 ['Breaklines candidates','troncons_visibles + raccords_souterrains','à valider','Les raccords sont fonctionnels, géométrie de conduite inconnue'],
 ['Occupation du sol','geometries_preparatoires.gpkg:mos_2025','prêt pour reclassement','Aucun coefficient de Manning attribué'],
 ['Pluie historique','pluie_observee_gometz_max24h.csv','prêt comme série observée','Station distante de 6,58 km; événement non associé à une période de retour'],
 ['Réseau pluvial 1D','absent','bloquant pour couplage','Diamètres, radiers, pentes, regards, avaloirs et capacités requis'],
 ['Infiltration','absente','à paramétrer','Sols à 1:250 000 insuffisants pour calibration locale'],
 ['Rugosité','absente','à calibrer','Ne pas déduire mécaniquement Manning du MNH'],
 ['Conditions aval','absentes','à définir','Chronique/niveau de l’Yvette requis pour événement simulé'],
 ['Validation','absente','bloquant pour validation','Repères, photos géolocalisées, emprises ou capteurs événementiels requis']
],columns=['composant','fichier_ou_couche','statut','limite_ou_action'])
hec_manifest.to_csv(hec/'manifest_hec_ras.csv',index=False)

# COG de publication pour les couches clés
cogdir=webdir/'cog'; cogdir.mkdir(parents=True,exist_ok=True)
for name in ['mnt_lidar_0p50m_bassin_plus100m','pente_degres','ombrage_315_45','aire_contributive_d8_m2','profondeur_remplissage_depressions_m']:
    src=rasdir/f'{name}.tif'; dst=cogdir/f'{name}.tif'
    rio_copy(src,dst,driver='COG',compress='DEFLATE',blocksize=512,overview_resampling='average')
    with rasterio.open(dst) as ds: print(dst.name,ds.driver,ds.is_tiled,ds.overviews(1))

# Bâtiments prêts pour extrusion côté conversion 3D Tiles
b3=gpd.read_file(pkg,layer='batiments').to_crs(4326)
b3['height_m']=pd.to_numeric(b3.get('hauteur'),errors='coerce')
b3['height_source']='BD TOPO hauteur'
b3['height_m']=b3.height_m.where(b3.height_m>0)
b3['base_alt_m_ign69']=pd.to_numeric(b3.get('altitude_minimale_sol'),errors='coerce')
b3['color_metric']='dist_axe_brut_m'
b3.to_file(webdir/'batiments_extrusion.geojson',driver='GeoJSON')

# Catalogue de publication. URLs laissées en paramètres, pas de faux service.
layers=[
 {'id':'vatencul_mnt','title':'MNT LiDAR HD 0,50 m','kind':'raster','file':'cog/mnt_lidar_0p50m_bassin_plus100m.tif','service':'WMS/WMTS'},
 {'id':'vatencul_slope','title':'Pente (degrés)','kind':'raster','file':'cog/pente_degres.tif','service':'WMS/WMTS'},
 {'id':'vatencul_hillshade','title':'Ombrage','kind':'raster','file':'cog/ombrage_315_45.tif','service':'WMS/WMTS'},
 {'id':'vatencul_uparea','title':'Aire contributive D8 (m²)','kind':'raster','file':'cog/aire_contributive_d8_m2.tif','service':'WMS/WMTS'},
 {'id':'vatencul_filldepth','title':'Profondeur de remplissage topographique (m)','kind':'raster','file':'cog/profondeur_remplissage_depressions_m.tif','service':'WMS/WMTS'},
 {'id':'vatencul_objects','title':'Objets Vatencul','kind':'vector','file':'../vectors/vatencul_objets_websig.gpkg','service':'WFS/OGC API Features'},
 {'id':'vatencul_buildings3d','title':'Bâtiments extrudables','kind':'3d-prep','file':'batiments_extrusion.geojson','service':'conversion 3D Tiles requise'}]
(webdir/'layer_catalog.json').write_text(json.dumps(layers,ensure_ascii=False,indent=2),encoding='utf-8')
print('HEC files',len(list(hec.iterdir())),'Web files',sum(1 for _ in webdir.rglob('*') if _.is_file()))

In [ ]:
from docx import Document
p=root/'Memoire_NAOMIS_Final_V3_revise.docx'
d=Document(p)
for i,para in enumerate(d.paragraphs):
    t=para.text.strip()
    if t.startswith('CHAPITRE 5') or t.startswith('CHAPITRE 6') or t.startswith('5.'):
        if i>500: print(i,para.style.name,repr(t[:180]))

In [ ]:
from docx import Document
p=root/'Memoire_NAOMIS_Final_V3_revise.docx'; d=Document(p)
for i in range(647,710):
    t=d.paragraphs[i].text.strip()
    if t: print(i,d.paragraphs[i].style.name,repr(t[:300]))

In [ ]:
from docx import Document
from docx.shared import Inches, Pt
from docx.enum.text import WD_ALIGN_PARAGRAPH
from docx.oxml import OxmlElement
from docx.oxml.ns import qn
from pathlib import Path
import shutil, re

src=root/'Memoire_NAOMIS_Final_V3_revise.docx'
dst=out/'Memoire_NAOMIS_Vatencul_LiDAR_WebSIG_revise.docx'
doc=Document(src)

# Helpers to create elements at the end then move them before a reference paragraph.
def move_before(element, ref_para):
    ref_para._p.addprevious(element)

def add_p_before(ref, text='', style=None, bold_prefix=None):
    p=doc.add_paragraph(style=style)
    if bold_prefix and text.startswith(bold_prefix):
        r=p.add_run(bold_prefix); r.bold=True; p.add_run(text[len(bold_prefix):])
    else: p.add_run(text)
    move_before(p._p,ref)
    return p

def add_heading_before(ref,text,style):
    return add_p_before(ref,text,style=style)

def add_picture_before(ref,path,width=6.2,caption=None):
    p=doc.add_paragraph(); p.alignment=WD_ALIGN_PARAGRAPH.CENTER
    p.add_run().add_picture(str(path),width=Inches(width)); move_before(p._p,ref)
    if caption:
        cp=doc.add_paragraph(caption,style='NAOMIS - Légende figure' if 'NAOMIS - Légende figure' in [s.name for s in doc.styles] else None)
        cp.alignment=WD_ALIGN_PARAGRAPH.CENTER; move_before(cp._p,ref)

def add_table_before(ref, headers, rows, widths=None):
    t=doc.add_table(rows=1,cols=len(headers)); t.style='Table Grid'
    for j,h in enumerate(headers):
        t.rows[0].cells[j].text=str(h)
        for r in t.rows[0].cells[j].paragraphs[0].runs:r.bold=True
    for row in rows:
        cells=t.add_row().cells
        for j,v in enumerate(row): cells[j].text=str(v)
    move_before(t._tbl,ref)
    return t

# Replace chapter 5 in document body.
paras=doc.paragraphs
idx5=next(i for i,p in enumerate(paras) if p.text.strip().startswith('CHAPITRE 5 -'))
idx6=next(i for i,p in enumerate(paras) if p.text.strip().startswith('CHAPITRE 6 -'))
ref=paras[idx6]
for p in paras[idx5:idx6]:
    p._element.getparent().remove(p._element)

add_heading_before(ref,'CHAPITRE 5 - CAS D’USAGE APPROFONDI : LIDAR, 3D ET RUISSELLEMENT DANS UN WEBSIG MAPSTORE','NAOMIS - Chapitre')
add_heading_before(ref,'5.1. Résultat principal et périmètre','NAOMIS - Section')
add_p_before(ref,"La délimitation contrôlée du bassin versant du Vatencul couvre 73,061 ha. Elle dépasse donc le seuil opérationnel de 70 ha fixé pour cette étude. La poursuite de la délimitation vers l’aval n’a pas été déclenchée : l’exutoire retenu, X = 643 225,25 m et Y = 6 844 872,75 m en Lambert-93, se situe à 3,54 m du raccord avec l’Yvette et évite d’intégrer le bassin amont de cette dernière.")
add_p_before(ref,"Le LiDAR est utilisé ici pour préparer trois familles de produits : un diagnostic topographique, des entrées auditables pour une future modélisation hydraulique et un corpus de publication 2D/3D. Aucun débit, aucune hauteur d’eau, aucune vitesse et aucune probabilité d’inondation ne sont calculés. MapStore reste un outil d’exploration et de communication des résultats produits en amont, non un moteur hydraulique.")

add_heading_before(ref,'5.2. Données effectivement mobilisées','NAOMIS - Section')
add_p_before(ref,"Le traitement repose sur les seize dalles MNT, MNS et MNH LiDAR HD IGN à 0,50 m, le bassin et ses raccords fonctionnels documentés, la BD TOPO 2026, le MOS Île-de-France 2025, le PLU approuvé le 10 avril 2025, le PPRI de l’Yvette, l’imperméabilisation Copernicus 2024 et les précipitations Météo-France à six minutes. Toutes les géométries de travail sont harmonisées en EPSG:2154 et les altitudes du terrain restent référencées à IGN69.")
add_table_before(ref,['Donnée','Usage dans le cas','Limite principale'],[
 ['MNT/MNS/MNH LiDAR HD, 0,50 m','Relief, pente, hauteur des objets, terrain 3D','Classification et ouvrages enterrés à contrôler'],
 ['BD TOPO 2026','Bâtiments, routes, ponts, végétation, hydrographie','Niveaux de plancher et réseau EP absents'],
 ['MOS 2025 et Copernicus 2024','Occupation du sol et imperméabilisation','Ne fournit pas directement infiltration ou Manning'],
 ['PLU et PPRI','Contexte réglementaire et enjeux','PPRI de l’Yvette, pas une carte de ruissellement'],
 ['Pluie Météo-France 6 min','Hyétogramme historique préparatoire','Station la plus proche à 6,58 km'],
 ['Plans d’assainissement PDF','Contrôle documentaire possible','Pas de réseau vectoriel avec radiers et diamètres']])

add_heading_before(ref,'5.3. Contrôle de la délimitation et terrain de référence','NAOMIS - Section')
add_p_before(ref,"Le GeoPackage de référence contient un polygone valide de 730 611,75 m², un exutoire, sept tronçons visibles du Vatencul, sept raccords fonctionnels correspondant aux interruptions souterraines et un chemin aval contrôlé. Le polygone couvre l’exutoire, situé à 0,25 m de sa limite. Sa superficie est également cohérente avec les 2 922 447 cellules contributives de 0,25 m² enregistrées dans la couche d’exutoire.")
add_p_before(ref,"Le MNT brut découpé sur le bassin présente une altitude comprise entre 49,91 et 165,61 m, une médiane de 145,43 m et une moyenne de 128,83 m. La pente calculée sur la grille de 0,50 m a une médiane de 5,93° et une moyenne de 10,28°. Les valeurs extrêmes locales, jusqu’à 83,46°, peuvent correspondre à des ruptures de terrain ou à des artefacts et ne sont pas assimilées sans contrôle à des talus hydrauliques.")
add_picture_before(ref,figdir/'relief_hypsometrie_pentes.png',6.4,'Figure 5.1 - Distribution de l’altitude et des pentes dans le bassin du Vatencul.')

add_heading_before(ref,'5.4. Diagnostic topographique reproductible','NAOMIS - Section')
add_p_before(ref,"Un diagnostic D8 distinct de la délimitation officielle a été recalculé sur le MNT brut après remplissage généralisé des dépressions. Avec un seuil d’aire contributive de 5 000 m², il produit 11,411 km d’axes. Il place 119 des 398 bâtiments du bassin à 10 m ou moins d’un axe et recoupe 66 des 183 tronçons routiers. Ces valeurs ne remplacent pas les résultats hydroconditionnés antérieurs : elles constituent un état brut reproductible destiné à localiser les secteurs où les buses, fossés, murs, bordures et passages enterrés doivent être vérifiés.")
add_p_before(ref,"Le remplissage relève 350 123 cellules, soit 11,98 % des cellules du bassin. Parmi les cellules relevées, la profondeur médiane est de 0,079 m et le 95e percentile de 2,974 m. Le volume intégré de 42 314 m³ est un volume géométrique de remplissage, non un volume de stockage mobilisable. Après filtrage à une profondeur minimale de 0,10 m et une aire minimale de 10 m², 188 dépressions candidates sont conservées pour inspection. Leur volume de remplissage cumulé atteint 40 673 m³.")
add_p_before(ref,"Le diagnostic brut échoue au point exutoire documenté : l’aire contributive recalculée y est presque nulle, alors que la couche hydroconditionnée de référence enregistre 73,061 ha. Ce résultat négatif est utile. Il démontre quantitativement que le MNT brut et un simple remplissage ne restituent pas la continuité souterraine du Vatencul. Les raccords fonctionnels doivent donc être conservés, documentés et remplacés par la géométrie réelle des ouvrages dès qu’elle sera disponible.")
add_picture_before(ref,figdir/'carte_diagnostic_vatencul.png',6.4,'Figure 5.2 - Relief, axes D8 bruts, dépressions candidates et continuités documentées.')

add_heading_before(ref,'5.5. Occupation du sol, exposition et contexte réglementaire','NAOMIS - Section')
add_p_before(ref,"Le MOS 2025 décrit un bassin mixte. Les bois et forêts occupent 20,91 ha, soit 28,62 % du bassin. Ils sont suivis par les surfaces engazonnées entretenues, 11,33 ha (15,51 %), les zones d’activités économiques, 9,16 ha (12,54 %), l’habitat individuel, 8,65 ha (11,84 %) et les voies routières, 7,99 ha (10,94 %). Le raster Copernicus indique une imperméabilisation moyenne de 29,91 % sur 7 320 pixels valides, avec une médiane nulle et 13,22 % des pixels à au moins 80 %.")
add_p_before(ref,"Le PPRI de l’Yvette n’intersecte que 0,140 ha du bassin, dont 0,139 ha en zone orange. Cette faible intersection ne permet pas de conclure à une faible exposition au ruissellement : le PPRI réglemente principalement le débordement de l’Yvette. De même, la proximité d’un bâtiment avec un axe topographique ne prouve ni l’entrée d’eau ni le dépassement d’un niveau de plancher.")
add_picture_before(ref,figdir/'occupation_sol_mos2025.png',6.4,'Figure 5.3 - Principales occupations du sol dans le bassin, MOS 2025.')

add_heading_before(ref,'5.6. Pluie historique préparée pour un futur modèle','NAOMIS - Section')
add_p_before(ref,"La station à six minutes la plus proche est Gometz-le-Châtel (91275001), à 6,58 km du centroïde du bassin. Sa série disponible s’étend du 3 mai 2018 au 21 août 2026 et comporte 99,19 % de valeurs RR renseignées. Les maxima glissants observés, calculés uniquement sur des fenêtres complètes, sont de 12,3 mm en 6 minutes, 35,8 mm en 1 heure, 41,6 mm en 3 heures, 44,2 mm en 6 heures et 75,2 mm en 24 heures.")
add_p_before(ref,"La fenêtre du 9 octobre 2024 à 05:36 UTC au 10 octobre à 05:30 UTC totalise 75,2 mm sans valeur manquante. Elle est exportée comme hyétogramme historique de démonstration. Elle n’est pas associée à une période de retour et ne suffit pas à calibrer un modèle local, notamment pour les orages convectifs dont la variabilité spatiale peut être forte à 6,58 km.")
add_picture_before(ref,figdir/'hyetogramme_gometz_2024_10_09.png',6.4,'Figure 5.4 - Hyétogramme historique observé préparé pour un futur essai hydraulique.')

add_heading_before(ref,'5.7. Préparation HEC-RAS sans simulation','NAOMIS - Section')
add_p_before(ref,"Le paquet préparatoire contient le MNT LiDAR brut, le bassin, l’exutoire, les tronçons visibles, les raccords souterrains documentés, les bâtiments, les routes, les ponts, le MOS 2025 et l’hyétogramme historique. Ces éléments permettent de créer un terrain RAS Mapper, d’esquisser un domaine 2D et de préparer des lignes de rupture candidates.")
add_p_before(ref,"La simulation ne peut pas être exécutée de façon scientifiquement défendable avec les seules données disponibles. Il manque le réseau pluvial vectoriel, les regards et avaloirs, les diamètres, radiers, pentes et états des conduites, les caractéristiques des buses, les conditions aval sur l’Yvette, des paramètres locaux d’infiltration et de rugosité, ainsi que des observations événementielles indépendantes. Aucun coefficient de Manning, aucune loi d’infiltration et aucune condition aux limites n’ont donc été inventés.")

add_heading_before(ref,'5.8. Paquet GeoServer et démonstrateur MapStore','NAOMIS - Section')
add_p_before(ref,"Cinq rasters ont été convertis en Cloud Optimized GeoTIFF tuilés avec niveaux d’aperçu : MNT, pente, ombrage, aire contributive et profondeur de remplissage. Un GeoPackage rassemble le bassin, les axes, les dépressions, les bâtiments, les routes, l’hydrographie, la végétation, le MOS, le PLU et le PPRI. Les 398 bâtiments sont aussi exportés en GeoJSON avec hauteur BD TOPO et altitude minimale du sol lorsqu’elles sont disponibles, afin de permettre une conversion ultérieure en 3D Tiles.")
add_p_before(ref,"Le démonstrateur proposé comprend une vue « Comprendre le terrain », une vue « Chemins de l’eau », une vue « Stockage et ouvrages », une vue « Exposition des bâtiments » et une vue « Préparer la simulation ». L’animation temporelle de hauteur ou de vitesse reste désactivée tant qu’un modèle dynamique validé n’a pas produit ces variables. La 3D sert à examiner le relief, les bâtiments et les ouvrages, sans être présentée comme une validation hydraulique.")
add_table_before(ref,['Vue','Couches disponibles','Fonction'],[
 ['Comprendre le terrain','MNT, ombrage, pente, bâti 3D','Inspection du relief et des ruptures'],
 ['Chemins de l’eau','Aire contributive, axes bruts, Vatencul, raccords','Comparer topographie brute et continuités documentées'],
 ['Stockage et ouvrages','188 dépressions, 4 ponts, routes','Prioriser les contrôles terrain'],
 ['Exposition','398 bâtiments, distance aux axes','Présélection sans conclure à l’inondation intérieure'],
 ['Préparer la simulation','Terrain, domaine, pluie, manifeste des lacunes','Organiser le passage vers HEC-RAS']])

add_heading_before(ref,'5.9. Synthèse et portée des résultats','NAOMIS - Section')
add_p_before(ref,"Le bassin de 73,061 ha satisfait le seuil demandé et n’a pas à être étendu. Les produits calculés permettent un diagnostic topographique détaillé, la préparation d’un modèle et un démonstrateur WebSIG crédible. La divergence entre l’exutoire officiel hydroconditionné et le calcul sur MNT brut établit que les ouvrages et continuités souterraines contrôlent le résultat. Les 119 bâtiments proches, 66 croisements routiers et 188 dépressions candidates définissent une liste d’inspection, pas une carte d’aléa.")

# Harmonise résumé FR/EN et les mentions de l'ancienne superficie dans la discussion.
for p in doc.paragraphs:
    t=p.text
    if 'L’analyse exécutée repose sur une emprise analytique définitive de 71,86975 ha' in t:
        p.text=t.replace('L’analyse exécutée repose sur une emprise analytique définitive de 71,86975 ha','La nouvelle application repose sur le bassin hydroconditionné contrôlé de 73,061 ha')
    if 'Le cas principal porte sur un bassin de 71,86975 ha.' in t:
        p.text=t.replace('Le cas principal porte sur un bassin de 71,86975 ha.','Le cas principal porte sur un bassin de 73,061 ha.')

# Ajoute un avertissement de révision au début du chapitre 6.
ch6=next(p for p in doc.paragraphs if p.text.strip().startswith('CHAPITRE 6 -'))
add_p_before(ch6,"Note de révision : le chapitre 5 remplace l’ancienne application par le cas d’usage LiDAR–3D–ruissellement pour MapStore. Les résultats de sensibilité historique discutés ci-après restent des éléments de comparaison méthodologique; les livrables recalculés et préparés sont ceux du nouveau chapitre 5.",style=None)

doc.save(dst)
print(dst, dst.stat().st_size)
print('paragraphes',len(Document(dst).paragraphs),'tables',len(Document(dst).tables),'images',len(Document(dst).inline_shapes))

In [ ]:
from pathlib import Path
import json, hashlib, os, subprocess, shutil, pandas as pd, geopandas as gpd, rasterio, fiona
out=Path('/workspace/vatencul_lidar_websig_outputs'); webdir=out/'mapstore_geoserver'; docdir=out/'documentation'; pkg=out/'vectors/vatencul_objets_websig.gpkg'

# Configuration MapStore modèle avec URL explicites à remplacer
cfg={
 "version":2,
 "map":{"projection":"EPSG:3857","center":{"x":2.23,"y":48.70,"crs":"EPSG:4326"},"zoom":14,
 "layers":[
  {"id":"osm","type":"osm","title":"OpenStreetMap","name":"mapnik","group":"Fond","visibility":True},
  {"id":"vatencul_hillshade","type":"wms","url":"${GEOSERVER_WMS_URL}","name":"vatencul:ombrage_315_45","title":"Ombrage LiDAR","group":"Comprendre le terrain","visibility":True,"opacity":0.65},
  {"id":"vatencul_slope","type":"wms","url":"${GEOSERVER_WMS_URL}","name":"vatencul:pente_degres","title":"Pente (degrés)","group":"Comprendre le terrain","visibility":False},
  {"id":"vatencul_uparea","type":"wms","url":"${GEOSERVER_WMS_URL}","name":"vatencul:aire_contributive_d8_m2","title":"Aire contributive D8 (m²)","group":"Chemins de l’eau","visibility":False},
  {"id":"vatencul_filldepth","type":"wms","url":"${GEOSERVER_WMS_URL}","name":"vatencul:profondeur_remplissage_depressions_m","title":"Profondeur de remplissage topographique (m)","group":"Stockage et ouvrages","visibility":False},
  {"id":"vatencul_basin","type":"wms","url":"${GEOSERVER_WMS_URL}","name":"vatencul:bassin","title":"Bassin du Vatencul","group":"Chemins de l’eau","visibility":True},
  {"id":"vatencul_axes","type":"wms","url":"${GEOSERVER_WMS_URL}","name":"vatencul:axes_concentration_brut_5000m2","title":"Axes topographiques bruts","group":"Chemins de l’eau","visibility":True},
  {"id":"vatencul_depressions","type":"wms","url":"${GEOSERVER_WMS_URL}","name":"vatencul:depressions_candidates","title":"Dépressions candidates","group":"Stockage et ouvrages","visibility":False},
  {"id":"vatencul_buildings","type":"wms","url":"${GEOSERVER_WMS_URL}","name":"vatencul:batiments","title":"Bâtiments","group":"Exposition","visibility":False},
  {"id":"vatencul_3dtiles","type":"3dtiles","url":"${CESIUM_3DTILES_URL}/tileset.json","title":"Bâtiments 3D","group":"Scène 3D","visibility":False}
 ]},
 "catalogServices":{"services":{"vatencul-wms":{"url":"${GEOSERVER_WMS_URL}","type":"wms","title":"GeoServer Vatencul"}}},
 "widgetsConfig":{"widgets":[]}
}
(webdir/'mapstore_config_template.json').write_text(json.dumps(cfg,ensure_ascii=False,indent=2),encoding='utf-8')

# README scientifique et technique
metrics=pd.read_csv(out/'tables/indicateurs_territoriaux.csv').iloc[0]
readme=f"""# Paquet Vatencul LiDAR, ruissellement et WebSIG

## Résultat de délimitation

Le bassin contrôlé couvre **73,061 ha**. Le seuil demandé de 70 ha est dépassé; aucune extension aval n'a été appliquée. L'exutoire se situe en Lambert-93 à X=643225,25 m, Y=6844872,75 m, juste avant le raccord avec l'Yvette.

## Produits

- `Memoire_NAOMIS_Vatencul_LiDAR_WebSIG_revise.docx` : mémoire dont le chapitre 5 a été remplacé.
- `rasters/` : MNT, MNS, MNH, pente, aspect, ombrage, courbure, D8, aire contributive et profondeur de remplissage.
- `vectors/vatencul_objets_websig.gpkg` : bassin, axes, dépressions, bâtiments, routes, hydrographie, végétation, MOS, PLU et PPRI.
- `hec_ras_preparation/` : terrain, géométries et pluie préparés; aucune simulation exécutée.
- `mapstore_geoserver/` : COG, bâtiments extrudables, catalogue et configuration MapStore modèle.
- `tables/` et `figures/` : résultats quantitatifs et illustrations.

## Résultats calculés

- Axe D8 brut au seuil de 5 000 m² : **{metrics.axes_brut_longueur_km:.3f} km**.
- Bâtiments dans le bassin : **{int(metrics.batiments_bassin_n)}**; à 10 m ou moins d'un axe brut : **{int(metrics.batiments_proches_axes_brut_10m_n)}**.
- Tronçons routiers dans le bassin : **{int(metrics.routes_bassin_n)}**; croisant un axe brut : **{int(metrics.troncons_routes_croisant_axes_brut_n)}**.
- Dépressions candidates : **{int(metrics.depressions_candidates_n)}**.

## Interprétation obligatoire

Les axes sont des concentrations topographiques, pas des débits. Les dépressions sont issues d'un remplissage géométrique, pas des volumes de stockage hydraulique. La proximité d'un bâtiment ne prouve pas son inondation. Le PPRI de l'Yvette ne cartographie pas le ruissellement du Vatencul.

Le calcul brut ne restitue pas l'aire contributive au point exutoire documenté. Le bassin officiel repose sur les raccords fonctionnels hydroconditionnés. Cette divergence montre que les buses et continuités enterrées doivent être confirmées avant modélisation.

## Blocages pour HEC-RAS

Réseau EP vectoriel complet, diamètres, radiers, pentes, regards, avaloirs, géométrie et état des buses, conditions aval sur l'Yvette, infiltration et rugosité calibrées, puis observations événementielles indépendantes. Aucun paramètre absent n'a été inventé.

## Déploiement

1. Publier les COG via GeoServer WMS/WMTS.
2. Importer les couches du GeoPackage dans PostGIS ou GeoServer.
3. Convertir `batiments_extrusion.geojson` en 3D Tiles avec un outil compatible.
4. Remplacer les variables `${{GEOSERVER_WMS_URL}}` et `${{CESIUM_3DTILES_URL}}` dans `mapstore_config_template.json`.
5. Ne créer les couches temporelles hauteur/vitesse qu'après une simulation validée.
"""
(out/'README.md').write_text(readme,encoding='utf-8')

# Inventaire et sommes SHA-256
records=[]
for p in sorted(out.rglob('*')):
    if p.is_file() and p.name!='manifest_sha256.csv':
        h=hashlib.sha256()
        with open(p,'rb') as f:
            for chunk in iter(lambda:f.read(1024*1024),b''): h.update(chunk)
        records.append({'path':str(p.relative_to(out)),'size_bytes':p.stat().st_size,'sha256':h.hexdigest()})
pd.DataFrame(records).to_csv(out/'manifest_sha256.csv',index=False)
print('fichiers inventoriés',len(records),'taille totale GiB',sum(x['size_bytes'] for x in records)/2**30)
print('couches GPKG',fiona.listlayers(pkg))

In [ ]:
from pathlib import Path
import rasterio, fiona, pandas as pd, geopandas as gpd, zipfile, os, json
from docx import Document
out=Path('/workspace/vatencul_lidar_websig_outputs')

# Vérifications finales
checks=[]
for p in out.rglob('*.tif'):
    try:
        with rasterio.open(p) as ds:
            checks.append((str(p.relative_to(out)),'raster_ok',ds.width,ds.height,ds.crs.to_string()))
    except Exception as e: checks.append((str(p.relative_to(out)),'raster_error',str(e),'',''))
for p in out.rglob('*.gpkg'):
    try: checks.append((str(p.relative_to(out)),'gpkg_ok',len(fiona.listlayers(p)),'',','.join(fiona.listlayers(p))))
    except Exception as e: checks.append((str(p.relative_to(out)),'gpkg_error',str(e),'',''))
docp=out/'Memoire_NAOMIS_Vatencul_LiDAR_WebSIG_revise.docx'
d=Document(docp)
text='\n'.join(p.text for p in d.paragraphs)
checks.append((docp.name,'docx_ok',len(d.paragraphs),len(d.tables),'chapitre5_nouveau='+str('CAS D’USAGE APPROFONDI' in text)))
pd.DataFrame(checks,columns=['fichier','statut','valeur1','valeur2','detail']).to_csv(out/'controle_integrite.csv',index=False)
errors=[x for x in checks if 'error' in x[1]]
print('contrôles',len(checks),'erreurs',len(errors),'nouveau chapitre', 'CAS D’USAGE APPROFONDI' in text)

# Archive sans dupliquer une archive antérieure
zip_path=Path('/workspace/vatencul_lidar_websig_livraison.zip')
if zip_path.exists(): zip_path.unlink()
with zipfile.ZipFile(zip_path,'w',compression=zipfile.ZIP_DEFLATED,compresslevel=6,allowZip64=True) as z:
    for p in sorted(out.rglob('*')):
        if p.is_file(): z.write(p,arcname=str(Path(out.name)/p.relative_to(out)))
print('archive',zip_path,'taille MiB',round(zip_path.stat().st_size/2**20,1))